# Demo Reset Utility

This notebook prepares a completely clean demonstration environment.

It will:

1. Back up the finance database.
2. Archive the previous `demo_session.json`.
3. Delete all records from the six demo business tables.
4. Preserve the database structure.
5. Reset SQLite sequence counters.
6. Validate database integrity.
7. Confirm that all demo tables are empty.

> This utility removes evidence from all previous local demo cases.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import shutil
import sqlite3

PROJECT_ROOT = Path(
    "/Users/pmayank/workspace/LdcDemo"
).resolve()

DATABASE_PATH = PROJECT_ROOT / "data" / "finance_demo.db"
SESSION_PATH = PROJECT_ROOT / "data" / "demo_session.json"
BACKUP_DIRECTORY = PROJECT_ROOT / "data" / "demo_backups"

DEMO_TABLES = [
    "business_audit_events",
    "posting_records",
    "approval_requests",
    "finance_decisions",
    "event_receipts",
    "invoice_cases",
]

RESET_CONFIRMATION = "RESET_ALL_DEMO_DATA"

assert RESET_CONFIRMATION == "RESET_ALL_DEMO_DATA", (
    "Reset confirmation is not enabled."
)

assert DATABASE_PATH.exists(), (
    f"Database not found: {DATABASE_PATH}"
)

timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

BACKUP_DIRECTORY.mkdir(parents=True, exist_ok=True)

database_backup_path = (
    BACKUP_DIRECTORY
    / f"finance_demo_before_reset_{timestamp}.db"
)

session_backup_path = (
    BACKUP_DIRECTORY
    / f"demo_session_before_reset_{timestamp}.json"
)

print("=" * 72)
print("FULL DEMO RESET")
print("=" * 72)
print(f"Database : {DATABASE_PATH}")
print(f"Backup   : {database_backup_path}")
print("-" * 72)

# Create a consistent SQLite backup.
with sqlite3.connect(DATABASE_PATH) as source_connection:
    with sqlite3.connect(database_backup_path) as backup_connection:
        source_connection.backup(backup_connection)

print("✅ Database backup created")

# Archive the previous session instead of permanently deleting it.
if SESSION_PATH.exists():
    shutil.move(SESSION_PATH, session_backup_path)
    print(f"✅ Previous session archived: {session_backup_path}")
else:
    print("ℹ️ No previous demo session was present")

before_counts = {}
after_counts = {}

with sqlite3.connect(DATABASE_PATH) as connection:
    connection.execute("PRAGMA foreign_keys = ON")

    available_tables = {
        row[0]
        for row in connection.execute(
            """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            """
        ).fetchall()
    }

    missing_tables = [
        table_name
        for table_name in DEMO_TABLES
        if table_name not in available_tables
    ]

    if missing_tables:
        raise RuntimeError(
            "Reset stopped because required tables are missing: "
            + ", ".join(missing_tables)
        )

    for table_name in DEMO_TABLES:
        before_counts[table_name] = connection.execute(
            f'SELECT COUNT(*) FROM "{table_name}"'
        ).fetchone()[0]

    try:
        connection.execute("BEGIN IMMEDIATE")

        # Delete dependent evidence first and invoice cases last.
        for table_name in DEMO_TABLES:
            connection.execute(
                f'DELETE FROM "{table_name}"'
            )

        # Reset auto-increment counters when sqlite_sequence exists.
        sequence_table_exists = connection.execute(
            """
            SELECT COUNT(*)
            FROM sqlite_master
            WHERE type = 'table'
              AND name = 'sqlite_sequence'
            """
        ).fetchone()[0]

        if sequence_table_exists:
            placeholders = ",".join(
                "?" for _ in DEMO_TABLES
            )

            connection.execute(
                f"""
                DELETE FROM sqlite_sequence
                WHERE name IN ({placeholders})
                """,
                DEMO_TABLES,
            )

        connection.commit()

    except Exception:
        connection.rollback()
        raise

    for table_name in DEMO_TABLES:
        after_counts[table_name] = connection.execute(
            f'SELECT COUNT(*) FROM "{table_name}"'
        ).fetchone()[0]

    integrity_result = connection.execute(
        "PRAGMA integrity_check"
    ).fetchone()[0]

print("\n" + "=" * 72)
print("RESET RESULTS")
print("=" * 72)
print(f"{'Table':<32}{'Before':>12}{'After':>12}")
print("-" * 56)

for table_name in DEMO_TABLES:
    print(
        f"{table_name:<32}"
        f"{before_counts[table_name]:>12,}"
        f"{after_counts[table_name]:>12,}"
    )

all_tables_empty = all(
    count == 0
    for count in after_counts.values()
)

database_integrity_ok = integrity_result.lower() == "ok"
reset_successful = all_tables_empty and database_integrity_ok

print("-" * 72)
print(f"Database integrity : {integrity_result}")
print(f"Session removed    : {not SESSION_PATH.exists()}")
print(f"Backup available   : {database_backup_path.exists()}")
print("-" * 72)

if reset_successful:
    print("✅ FULL DEMO RESET COMPLETED")
    print("✅ ALL BUSINESS TABLES ARE EMPTY")
    print("✅ DATABASE SCHEMA HAS BEEN PRESERVED")
else:
    print("❌ DEMO RESET VALIDATION FAILED")

assert reset_successful, (
    "Reset did not complete successfully. "
    "The pre-reset database backup remains available."
)

FULL DEMO RESET
Database : /Users/pmayank/workspace/LdcDemo/data/finance_demo.db
Backup   : /Users/pmayank/workspace/LdcDemo/data/demo_backups/finance_demo_before_reset_20260914T114212Z.db
------------------------------------------------------------------------
✅ Database backup created
ℹ️ No previous demo session was present

RESET RESULTS
Table                                 Before       After
--------------------------------------------------------
business_audit_events                      9           0
posting_records                            0           0
approval_requests                          0           0
finance_decisions                          0           0
event_receipts                             5           0
invoice_cases                              5           0
------------------------------------------------------------------------
Database integrity : ok
Session removed    : True
Backup available   : True
----------------------------------------------------